# 08 · Final tables and figures

Builds the tables that combine outputs of earlier notebooks and draws every figure in the paper. No model is refitted here except the bootstrap for the Table 3 confidence intervals.

**Reads (`DATA_DIR`):** `pred_cohort`, `pred_features` (03); `pred_oof_90d`, `pred_split_90d` (04); `landmark_build` (02); `analysis_frame` (06); BP readings from MIMIC-IV `omr`

**Reads (`results/`):** JSON and CSV outputs of notebooks 01–06b

**Produces (aggregate, `results/`):** `table1.csv`, `tableS4.csv`, `table3_with_ci.csv`, `tableS9_denominators.csv`, `paper_tables.xlsx`, `08_key_numbers.json`, and `figures/` (Figures 1, 2A, 2B, 3, S1, S2, S3 as PNG and PDF)

**Paper items:** Table 1; Table 3 (confidence intervals); Table S4; Table S9; Figures 1–3 and S1–S3.

**Runtime:** about 10–20 minutes (mostly the Table 3 bootstrap). Run notebooks 01–06b first.

## 0 · Setup

In [ ]:
# ---- edit these ----
REPO_DIR   = "/content/drive/MyDrive/Ronveer Sandhu/Project 1 -MIMIC analysis-hypertension retention/htn-bp-followup-mimic"
MIMIC_HOSP = "/content/drive/MyDrive/MIMIC Data/mimic-iv-2.2/hosp"
DATA_DIR   = "/content/drive/MyDrive/htn_bp_data"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!pip -q install duckdb==1.3.2 scikit-learn==1.6.1 statsmodels==0.15.0
!apt-get -qq install -y fonts-liberation > /dev/null

import os, sys, json, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from htn_utils import *
from landmark_models import ORDER, set_class_order

import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.patches import FancyBboxPatch, Rectangle

RESULTS_DIR = os.path.join(REPO_DIR, "results")
FIG_DIR = os.path.join(RESULTS_DIR, "figures"); os.makedirs(FIG_DIR, exist_ok=True)
rj = lambda n: json.load(open(os.path.join(RESULTS_DIR, f"{n}.json")))
rc = lambda n: pd.read_csv(os.path.join(RESULTS_DIR, f"{n}.csv"))

for p in font_manager.findSystemFonts():
    if "Liberation" in p: font_manager.fontManager.addfont(p)
plt.rcParams.update({"font.family": "Liberation Sans", "font.size": 9, "pdf.fonttype": 42})
def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIG_DIR, f"{name}.{ext}"), dpi=300, bbox_inches="tight")
    plt.show(); print("saved", name)

J1, J2, J3 = rj("01_cohort_counts"), rj("02_landmark_counts"), rj("03_prediction_counts")
J4, J5, J6, J7 = rj("04_prediction_models"), rj("05_trajectories"), rj("06_main"), rj("07_multimorbidity")
set_class_order(J5["class_order"])

P   = load_df(DATA_DIR, "pred_cohort")
Fe  = load_df(DATA_DIR, "pred_features")
oof = load_df(DATA_DIR, "pred_oof_90d")
spl = load_df(DATA_DIR, "pred_split_90d")
if "subject_id" not in spl.columns:
    spl = spl.rename(columns={spl.columns[0]: "subject_id"})
lb  = load_df(DATA_DIR, "landmark_build")
A   = load_df(DATA_DIR, "analysis_frame")

S = (oof[["subject_id", "y"]].merge(spl[["subject_id", "set"]], on="subject_id", validate="1:1")
                             .merge(Fe, on="subject_id", validate="1:1"))
assert (S.subject_id.values == oof.subject_id.values).all()
assert (S.y == S.target_90).all()
check("Prediction sample", len(S), 30942)
check("  no recorded BP within 90 days", int(S.y.sum()), 10899)

## 1 · Table 1 and Table S4: the prediction sample

BP readings use the same definition as notebooks 01 and 03 (systolic 60–300 and diastolic 30–200 mmHg). The **qualifying reading** is the first reading with systolic pressure above 140 mmHg on the prediction date (`omr` sequence order). **Earlier readings** are the mean of all readings before the prediction date. Other rows use the features of notebook 03. Summaries use observed values; SMD compares patients with and without a recorded BP within 90 days.

In [ ]:
con = connect(MIMIC_HOSP)
pid = P[["subject_id", "index_date", "idx_day"]].copy()
pid["index_date"] = pd.to_datetime(pid.index_date).dt.date
con.register("pid", pid)
bp = con.execute('''
  SELECT o.subject_id, o.seq_num,
         DATE_DIFF('day', CAST(p.index_date AS DATE), CAST(o.chartdate AS DATE)) AS dd,
         TRY_CAST(SPLIT_PART(o.result_value, '/', 1) AS INTEGER) AS sbp,
         TRY_CAST(SPLIT_PART(o.result_value, '/', 2) AS INTEGER) AS dbp
  FROM omr o JOIN pid p ON p.subject_id = o.subject_id
  WHERE o.result_name LIKE 'Blood Pressure%' ''').df()
bp = bp[bp.sbp.between(60, 300) & bp.dbp.between(30, 200)].merge(pid[["subject_id", "idx_day"]], on="subject_id")
bp["rel"] = bp.dd - bp.idx_day                     # days from the prediction date

ids = S.subject_id.values
d0 = bp[(bp.rel == 0) & (bp.sbp > 140)].sort_values(["subject_id", "seq_num"])
q  = d0.groupby("subject_id")[["sbp", "dbp"]].first().reindex(ids)
ea = bp[bp.rel < 0].groupby("subject_id")[["sbp", "dbp"]].mean().reindex(ids)
S["q_sbp"], S["q_dbp"], S["e_sbp"], S["e_dbp"] = q.sbp.values, q.dbp.values, ea.sbp.values, ea.dbp.values

assert S.q_sbp.notna().all() and S.q_sbp.min() > 140, "qualifying reading not found for every patient"
check("No BP readings before the prediction date", int(S.e_sbp.isna().sum()), 5928)
early_dbp = bp[(bp.dd >= 0) & (bp.rel < 0) & (bp.dbp >= 90) & (bp.sbp <= 140)].subject_id.unique()
print(f"qualifying readings with diastolic >= 90: {100 * (S.q_dbp >= 90).mean():.1f}%")
print(f"earlier reading after index with diastolic >= 90 and systolic <= 140: "
      f"{np.isin(ids, early_dbp).sum():,} ({100 * np.isin(ids, early_dbp).mean():.1f}%)")

In [ ]:
RACE = [("White", "White"), ("Black", "Black"), ("Hispanic", "Hispanic"), ("Asian", "Asian"), ("Other/Unknown", "Other or unknown")]
INS  = ["Medicare", "Medicaid", "Other"]

def npct(f):  return lambda d: f"{f(d).sum():,} ({100 * f(d).mean():.1f})"
def msd(v, r=1): return lambda d: f"{d[v].mean():.{r}f} ({d[v].std():.{r}f})"
def miqr(v, r=0): return lambda d: f"{d[v].median():.{r}f} ({d[v].quantile(.25):.{r}f}–{d[v].quantile(.75):.{r}f})"
def smd_c(v):  return lambda a, b: abs(a[v].mean() - b[v].mean()) / np.sqrt((a[v].var() + b[v].var()) / 2)
def smd_b(f):
    def g(a, b):
        p1, p2 = f(a).mean(), f(b).mean(); s = np.sqrt((p1 * (1 - p1) + p2 * (1 - p2)) / 2)
        return 0.0 if s == 0 else abs(p1 - p2) / s
    return g

ROWS = [("Patients, n", lambda d: f"{len(d):,}", None),
        ("No recorded BP within 90 days, n (%)", npct(lambda d: d.y == 1), None),
        ("Age, years, mean (SD)", msd("age_index"), smd_c("age_index")),
        ("Female, n (%)", npct(lambda d: d.gender == "F"), smd_b(lambda d: d.gender == "F")),
        ("Race and ethnicity, n (%)", None, None)]
ROWS += [(f"  {lab}", npct(lambda d, k=k: d.race == k), smd_b(lambda d, k=k: d.race == k)) for k, lab in RACE]
ROWS += [("Insurance, n (%)", None, None)]
ROWS += [(f"  {k}", npct(lambda d, k=k: d.insurance == k), smd_b(lambda d, k=k: d.insurance == k)) for k in INS]
ROWS += [("Clinical", None, None),
         ("Systolic BP at qualifying reading, mmHg, mean (SD)", msd("q_sbp"), smd_c("q_sbp")),
         ("Diastolic BP at qualifying reading, mmHg, mean (SD)", msd("q_dbp"), smd_c("q_dbp")),
         ("Systolic BP, earlier readings, mmHg, mean (SD)", msd("e_sbp"), smd_c("e_sbp")),
         ("Diastolic BP, earlier readings, mmHg, mean (SD)", msd("e_dbp"), smd_c("e_dbp")),
         ("eGFR, mL/min/1.73 m², median (IQR)", miqr("egfr"), smd_c("egfr")),
         ("HbA1c, %, median (IQR)", miqr("hba1c", 1), smd_c("hba1c")),
         ("BMI, kg/m², mean (SD)", msd("bmi"), smd_c("bmi")),
         ("Elixhauser score, median (IQR)", miqr("elixhauser_vw"), smd_c("elixhauser_vw")),
         ("Antihypertensive drugs, median (IQR)", miqr("n_antihtn_drugs"), smd_c("n_antihtn_drugs")),
         ("Contact history before prediction", None, None),
         ("Prior encounters, median (IQR)", miqr("n_prior_enc"), smd_c("n_prior_enc")),
         ("Any prior emergency visit, n (%)", npct(lambda d: d.n_ed_prior > 0), smd_b(lambda d: d.n_ed_prior > 0)),
         ("Days since last visit, median (IQR)", miqr("recency_days"), smd_c("recency_days")),
         ("Missing values, n (%)", None, None)]
ROWS += [(f"  {lab}", npct(lambda d, v=v: d[v].isna()), smd_b(lambda d, v=v: d[v].isna()))
         for v, lab in [("egfr", "eGFR"), ("hba1c", "HbA1c"), ("bmi", "BMI"),
                        ("recency_days", "Days since last visit"),
                        ("sbp_sd", "Systolic BP variability (fewer than two readings)")]]

def make_table(groups, with_smd):
    a, b = S[S.y == 1], S[S.y == 0]
    out = []
    for lab, f, s in ROWS:
        r = {"Characteristic": lab, **{g: ("" if f is None else f(d)) for g, d in groups.items()}}
        if with_smd:
            r["SMD"] = "" if s is None else f"{s(a, b):.2f}"
        out.append(r)
    return pd.DataFrame(out)

table1 = make_table({"Overall": S, "No recorded BP in 90 days": S[S.y == 1], "BP recorded in 90 days": S[S.y == 0]}, True)
table1 = table1[table1.Characteristic != "No recorded BP within 90 days, n (%)"]
tableS4 = make_table({"Training": S[S.set == "train"], "Validation": S[S.set == "val"], "Test": S[S.set == "test"]}, False)
pd.set_option("display.width", 250, "display.max_colwidth", 55)
print("TABLE 1"); print(table1.to_string(index=False))
print("\nTABLE S4"); print(tableS4.to_string(index=False))

## 2 · Table 3 with 95% confidence intervals

Out-of-fold predictions from notebook 04; the same single threshold flags the 20% of patients with the highest risk. AUROC uses raw scores; observed/predicted ratio and calibration slope use recalibrated scores. Confidence intervals from 500 bootstrap samples of all 30,942 patients (seed 42). The true-positive-rate differences and their intervals come from notebook 04.

In [ ]:
y, raw, cal = oof.y.values, oof.oof.values, oof.oof_cal.values
grp = pd.DataFrame({
    "Race": S.race.values, "Insurance": S.insurance.values,
    "Sex": S.gender.map({"F": "Female", "M": "Male"}).values,
    "Age": pd.cut(S.age_index, [0, 49, 64, 79, 200], labels=["<50", "50–64", "65–79", "≥80"]).astype(str).values})
GORDER = [("Race", ["White", "Black", "Hispanic", "Asian", "Other/Unknown"]),
          ("Insurance", ["Other", "Medicare", "Medicaid"]),
          ("Sex", ["Female", "Male"]), ("Age", ["<50", "50–64", "65–79", "≥80"])]
flag = raw >= np.quantile(raw, 0.80)

def slope(p, yy):
    p = np.clip(p, 1e-6, 1 - 1e-6); lp = np.log(p / (1 - p))
    return sm.GLM(yy, sm.add_constant(lp), family=sm.families.Binomial()).fit().params[1]

def stats(yy, rr, cc, ff):
    return {"AUROC": roc_auc_score(yy, rr), "Obs/pred": yy.mean() / cc.mean(), "Slope": slope(cc, yy),
            "Flagged": ff.mean(), "TPR": ff[yy == 1].mean(), "FPR": ff[yy == 0].mean(), "PPV": yy[ff].mean()}

point = {(d, l): stats(y[i], raw[i], cal[i], flag[i]) for d, levs in GORDER for l in levs for i in [grp[d].values == l]}
saved = rc("table3_subgroups_point")
for _, s in saved.iterrows():
    p = point[(s.group, s.level)]
    assert abs(p["AUROC"] - s.auroc) < 0.002 and abs(p["TPR"] - s.tpr) < 0.002, (s.group, s.level)
print("point estimates match notebook 04")

rng = np.random.default_rng(42)
boot = {k: [] for k in point}
for j in range(500):
    b = rng.integers(0, len(y), len(y))
    for d, levs in GORDER:
        g = grp[d].values[b]
        for l in levs:
            i = g == l
            boot[(d, l)].append(stats(y[b][i], raw[b][i], cal[b][i], flag[b][i]))
    if (j + 1) % 100 == 0: print(f"  {j + 1}/500")

rows = []
for (d, l), p in point.items():
    bs = pd.DataFrame(boot[(d, l)]); s = saved[(saved.group == d) & (saved.level == l)].iloc[0]
    r = {"Group": d, "Level": l, "n": int(s.n), "No recorded BP, %": round(100 * s.outcome_rate, 1)}
    for k, v in p.items():
        dp = 2 if k in ("Obs/pred", "Slope") else 3
        r[k] = f"{v:.{dp}f} ({bs[k].quantile(.025):.{dp}f}–{bs[k].quantile(.975):.{dp}f})"
    r["TPR difference"] = ("Reference" if pd.isna(s.tpr_diff)
                           else f"{s.tpr_diff:.3f} ({s.tpr_diff_lo:.3f} to {s.tpr_diff_hi:.3f})")
    rows.append(r)
table3 = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 30)
print(table3.to_string(index=False))

## 3 · Table S9: analysis samples and denominators

In [ ]:
R = []
def row(part, analysis, start, cover, rules, n, events):
    R.append({"Part": part, "Analysis": analysis, "Starting sample": start, "Coverage required": cover,
              "Survival, death, and follow-up rules": rules, "n": n, "Outcome events": events})

first, nodata, data_after = J3["first_sbp_gt_140"], J3["no_data_after_prediction"], J3["data_after_prediction"]
t90 = J3["target_90"]
row("Prediction", "No recorded BP within 90 days (primary)", f"{first:,} with a first systolic reading >140 mmHg",
    f"90 days after the prediction date ({nodata + t90['data_ended']:,} without: {nodata:,} with no data after it, "
    f"{t90['data_ended']:,} whose data ended within 90 days)", f"Died within 90 days excluded ({t90['died']:,})",
    f"{t90['n']:,}", f"{t90['no_bp']:,} with no recorded BP")
for H in [30, 180, 270, 365]:
    t = J3[f"target_{H}"]
    row("Prediction", f"No recorded BP within {H} days", f"{data_after:,} with data after the prediction date",
        f"{H} days ({t['data_ended']:,} without)", f"Died within {H} days excluded ({t['died']:,})",
        f"{t['n']:,}", f"{t['no_bp']:,}")
s5 = rc("tableS5a_alternative_targets").set_index("target").loc["No BP within 90 days, deaths before BP counted"]
row("Prediction", "No recorded BP within 90 days, deaths before BP counted", f"{data_after:,}", "90 days",
    "Death before any BP reading counted as an outcome", f"{int(s5.n):,}", f"{int(s5.outcomes):,}")

# death as a competing event: first of BP, death, or end of coverage within 90 days
t_bp = (P.next_bp_day - P.idx_day).where(lambda s_: s_ > 0)
t_d  = P.days_to_death - P.idx_day
t_a  = P.admin_day - P.idx_day
bp90 = t_bp.le(90) & (t_bp <= t_d.fillna(np.inf))
d90  = ~bp90 & t_d.le(90)
cens = ~bp90 & ~d90 & t_a.lt(90)
assert int(bp90.sum()) == J3["competing_risk_90"]["bp"] and int(d90.sum()) == J3["competing_risk_90"]["death_before_bp"]
row("Prediction", "Death as a competing event (Table S5B)", f"{first:,}",
    f"Any data after the prediction date ({nodata:,} without)",
    "Death before a BP reading is a competing event; censored at the end of coverage", f"{len(P):,}",
    f"BP {int(bp90.sum()):,}; death before BP {int(d90.sum()):,}; coverage ended first {int(cens.sum()):,}")

FU = "follow-up ends at the end of coverage, day 1,460, 365 days after the last hospital or emergency contact, or death"
elig, atidx = J1["eligible_at_index"], J1["at_index_cohort"]
pm, v2, pk = J2["primary_landmark"], J2["day730_information_sample"], J2["events_before_index_kept"]
overlap = int((lb.in_day730_info & lb.in_landmark).sum())
row("Association", "At-index cohort", f"{elig:,} eligible at index", f"Through day 730 ({elig - atidx:,} without)",
    "—", f"{atidx:,}", "—")
row("Association", "Landmark population (primary)", f"{atidx:,}", "Through day 730",
    "Alive at day 730, confirmed by any record after day 730; no CV hospitalization before index or during days 1–730; " + FU,
    f"{pm['n']:,}", f"{pm['events']:,} CV hospitalizations")
row("Association", "Day-730-information sample", f"{atidx:,}", "Through day 730",
    "Alive at day 730 (no recorded death by day 730), confirmed only by hospital or emergency contact during "
    f"days 366–730 (no later information); {overlap:,} also in the primary population; follow-up as primary",
    f"{v2['n']:,}", f"{v2['events']:,}")
row("Association", "Events before index kept", f"{atidx:,}", "Through day 730",
    f"As primary, but patients with a CV hospitalization before index kept (+{pk['n'] - pm['n']:,})",
    f"{pk['n']:,}", f"{pk['events']:,}")
s3 = rc("tableS3").set_index("analysis")
for key, lab, rules in [("Complete cases", "Complete case", "As primary; eGFR, glycated hemoglobin, and body mass index all observed"),
                        ("Censoring at the end of database coverage only", "Censoring at end of coverage only",
                         "As primary, but follow-up ends only at the end of coverage, day 1,460, or death")]:
    row("Association", lab, "Landmark population", "Through day 730", rules,
        f"{int(s3.loc[key, 'n']):,}", f"{int(s3.loc[key, 'events']):,}")

tableS9 = pd.DataFrame(R)
pd.set_option("display.max_colwidth", 80)
print(tableS9.to_string(index=False))

## 4 · Figure 1: study population and design

All numbers come from the saved outputs; the landmark exclusions are recomputed from `landmark_build` in the order shown.

In [ ]:
fl = [s["n"] for s in J1["flow"]]
N0, SEC, D90 = fl[1], fl[1] - fl[2], fl[2] - fl[3]          # adults with a later BP; correctable cause; death within 90 days
ELIG, ATIDX = J1["eligible_at_index"], J1["at_index_cohort"]; COV730 = ELIG - ATIDX

keep = pd.Series(True, index=lb.index); LM_EXCL = {}
for lab, m in [("Died by day 730",                  lb.status730 != "died by day 730"),
               ("Survival at 730 not confirmed",    lb.status730 != "survival unknown"),
               ("Alive at 730, no later record",    lb.status730 == "alive: record after day 730"),
               ("CV hospitalization, days 1–730",   ~lb.mace_day.between(1, 730)),
               ("CV hospitalization before index",  lb.prior_mace.fillna(0) == 0),
               ("No follow-up after day 730",       lb.fu_end > 730)]:
    LM_EXCL[lab] = int((keep & ~m).sum()); keep &= m
assert keep.equals(lb.in_landmark)
LM, LM_EV, V2, V2_EV = pm["n"], pm["events"], v2["n"], v2["events"]
SBP140, NODATA, CR = first, nodata, data_after
NO140 = ELIG - SBP140
END90, DIED90, PRED, NOBP = t90["data_ended"], t90["died"], t90["n"], t90["no_bp"]
PRED_DAY = int(S.idx_day.median())
assert N0 - SEC - D90 == ELIG and ATIDX - sum(LM_EXCL.values()) == LM
assert SBP140 - NODATA == CR and CR - END90 - DIED90 == PRED
print({"N0": N0, "SEC": SEC, "D90": D90, **LM_EXCL, "LM": LM, "V2": V2, "V2 also primary": overlap, "PRED": PRED})
f = lambda n: f"{n:,}"

W, H = 6.5, 5.8
fig = plt.figure(figsize=(W, H))
HA = 3.80
Ax = fig.add_axes([0, 1 - HA / H, 1, HA / H]); Ax.set_xlim(0, W); Ax.set_ylim(0, HA); Ax.axis("off")
EDGE, GREY = "#333333", "#f2f2f2"
def box(x, y_, w, h, t, fc="white", bold=False, fs=8, ls="-", ha="center"):
    Ax.add_patch(FancyBboxPatch((x, y_), w, h, boxstyle="round,pad=0,rounding_size=0.04", fc=fc, ec=EDGE, lw=0.9, ls=ls))
    Ax.text(x + w / 2 if ha == "center" else x + 0.07, y_ + h / 2, t, ha=ha, va="center",
            fontsize=fs, fontweight="bold" if bold else "normal", linespacing=1.25)
def seg(xs, ys, ls="-"): Ax.plot(xs, ys, color=EDGE, lw=0.9, ls=ls, solid_capstyle="butt")
def arr(x0, y0, x1, y1, ls="-"):
    Ax.annotate("", xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(arrowstyle="-|>", color=EDGE, lw=0.9, ls=ls, mutation_scale=8, shrinkA=0, shrinkB=0))

BW, GAP = 1.385, 0.3
X = [0.02 + i * (BW + GAP) for i in range(4)]; C = [x + BW / 2 for x in X]; MID = [X[i] + BW + GAP / 2 for i in range(3)]
Ax.text(0, HA, "A", fontsize=12, fontweight="bold", va="top")

R1, R1H = 2.08, 0.50
box(X[0], R1, BW, R1H, f"Adults with hypertension\nand a later BP reading\nn = {f(N0)}")
box(X[1], R1, BW, R1H, f"Eligible at index\nn = {f(ELIG)}", bold=True)
box(X[2], R1, BW, R1H, f"At-index cohort\n(coverage through\nday 730), n = {f(ATIDX)}")
box(X[3], R1, BW, R1H, f"Landmark population\n(primary), n = {f(LM)}\n{f(LM_EV)} CV hospitalizations",
    fc="#dce8f4", bold=True, fs=7.6)
ym = R1 + R1H / 2
for i in range(3): arr(X[i] + BW, ym, X[i + 1], ym)
EX_Y = R1 + R1H + 0.24
def excl_above(i, h, t, w=1.55, fs=7.2, ha="center", x0=None):
    seg([MID[i], MID[i]], [ym, EX_Y]); box(MID[i] - w / 2 if x0 is None else x0, EX_Y, w, h, t, fc=GREY, fs=fs, ha=ha)
excl_above(0, 0.42, f"Surgically correctable\nsecondary cause ({f(SEC)})\nDeath within 90 days ({f(D90)})")
excl_above(1, 0.32, f"Coverage ended before\nday 730 ({f(COV730)})")
excl_above(2, 0.88, "\n".join(f"{k} ({f(v)})" for k, v in LM_EXCL.items()), w=W - 4.5 - 0.02, ha="left", x0=4.5)
Ax.text(0.04, R1 + R1H + 0.05, "Association analysis", fontsize=8.5, fontstyle="italic", va="bottom")

V2Y = 1.32
box(X[3], V2Y, BW, 0.42, f"Day-730-information\nsample, n = {f(V2)}\n{f(V2_EV)} CV hospitalizations", ls="--", fs=7.4)
seg([C[2], C[2]], [R1, V2Y + 0.21], ls="--"); arr(C[2], V2Y + 0.21, X[3], V2Y + 0.21, ls="--")
Ax.text(C[2] + 0.08, V2Y + 0.25, "survival confirmed\nby day 730 only", fontsize=6.8, ha="left", va="bottom", color="#444444")

R2, R2H = 0.50, 0.56
Ax.text(0.04, R2 + R2H / 2, "Prediction analysis", fontsize=8.5, fontstyle="italic", va="center", ha="left")
box(X[1], R2, BW, R2H, f"First systolic reading\n>140 mmHg, n = {f(SBP140)}")
box(X[2], R2, BW, R2H, f"Any data after the\nprediction date, n = {f(CR)}\n(competing-event\nsample)", fs=7.4)
box(X[3], R2, BW, R2H, f"Prediction sample\nn = {f(PRED)}; {f(NOBP)} with\nno recorded BP in 90 days", fc="#e3f1e6", bold=True, fs=7.4)
seg([C[1] - 0.25, C[1] - 0.25], [R1, R2 + R2H + 0.6]); arr(C[1] - 0.25, R2 + R2H + 0.6, C[1] - 0.25, R2 + R2H)
seg([X[0] + BW - 0.05, C[1] - 0.25], [R2 + R2H + 0.32] * 2)
box(X[0] + 0.05, R2 + R2H + 0.16, BW - 0.1, 0.32, f"No systolic reading\n>140 mmHg ({f(NO140)})", fc=GREY, fs=7.2)
ym2 = R2 + R2H / 2
for i in (1, 2): arr(X[i] + BW, ym2, X[i + 1], ym2)
def excl_below(i, h, t, w=1.5):
    seg([MID[i], MID[i]], [ym2, R2 - 0.1]); box(MID[i] - w / 2, R2 - 0.1 - h, w, h, t, fc=GREY, fs=7.2)
excl_below(1, 0.32, f"No data after the\nprediction date ({f(NODATA)})")
excl_below(2, 0.32, f"Data ended within 90 days ({f(END90)})\nDied within 90 days ({f(DIED90)})", w=1.62)

B = fig.add_axes([0.25, 0.078, 0.72, 0.21])
fig.text(0, 0.318, "B", fontsize=12, fontweight="bold", va="top")
timing = [("Covariates", -365, 0, "#8c8c8c"), ("Trajectory classes", 91, 727, "#0072B2"),
          ("Gap in recorded contact", 0, 730, "#56B4E9"), ("Outcome follow-up", 730, 1460, "#D55E00"),
          (f"Prediction window (90 days)", PRED_DAY, PRED_DAY + 90, "#009E73")]
for i, (lab, a, b_, col) in enumerate(timing):
    B.add_patch(Rectangle((a, -i - 0.32), b_ - a, 0.64, color=col, lw=0))
    B.text(-420, -i, lab, ha="right", va="center", fontsize=8)
for xv, lab in [(0, "Index"), (730, "Landmark (day 730)")]:
    B.axvline(xv, color=EDGE, ls="--", lw=0.8); B.text(xv, 0.62, lab, ha="center", va="bottom", fontsize=8)
B.set_xlim(-380, 1470); B.set_ylim(-4.6, 0.6); B.set_yticks([])
B.set_xticks([-365, 0, 365, 730, 1095, 1460]); B.tick_params(labelsize=8, length=3)
B.set_xlabel("Days from index blood pressure reading", fontsize=8.5)
for s_ in ("top", "right", "left"): B.spines[s_].set_visible(False)
save_fig(fig, "figure1")

## 5 · Figure 2 and Figure S1: prediction

In [ ]:
MODELS = ["XGBoost", "Penalized logistic", "Two-variable"]
LABEL = {"XGBoost": "Gradient-boosted trees", "Penalized logistic": "Penalized logistic regression",
         "Two-variable": "Two-variable model"}
COL = {"XGBoost": "#0072B2", "Penalized logistic": "#E69F00", "Two-variable": "#009E73"}
def clean(ax): ax.spines[["top", "right"]].set_visible(False); ax.tick_params(labelsize=9, length=3)

# Figure 2A: decision curve (test set)
dca = rc("figure2a_decision_curve"); d = dca[dca.threshold <= 0.60]
fig, ax = plt.subplots(figsize=(5.5, 3.6))
for m in MODELS:
    ax.fill_between(d.threshold, d[f"lo_{m}"], d[f"hi_{m}"], color=COL[m], alpha=0.18, lw=0)
    ax.plot(d.threshold, d[f"nb_{m}"], color=COL[m], lw=1.8, label=LABEL[m])
ax.plot(d.threshold, d.treat_all, color="0.55", lw=1.3, ls="--", label="Contact everyone")
ax.axhline(0, color="0.2", lw=0.9, label="Contact no one")
ax.set_xlim(0.05, 0.60); ax.set_ylim(-0.03, 0.34)
ax.set_xlabel("Risk threshold for outreach"); ax.set_ylabel("Net benefit")
clean(ax); ax.legend(frameon=False, fontsize=8.5, loc="upper right"); plt.tight_layout()
save_fig(fig, "figure2a_decision_curve")

# Figure 2B: detection by group (out-of-fold, top 20% flagged)
det = rc("figure2b_detection"); overall = J4["overall_detection"]
heads = {"Race": "Race and ethnicity", "Insurance": "Insurance", "Sex": "Sex", "Age": "Age, years"}
fig, ax = plt.subplots(figsize=(5.5, 4.6)); ticks, labels, yv = [], [], 0
for dim, levs in GORDER:
    ax.text(-0.02, yv, heads[dim], transform=ax.get_yaxis_transform(), ha="right", va="center", fontsize=9, fontweight="bold")
    yv += 1
    for lev in levs:
        r = det[(det.dim == dim) & (det.level == lev)].iloc[0]
        ax.errorbar(r.tpr, yv, xerr=[[r.tpr - r.lo], [r.hi - r.tpr]], fmt="o", color="#0072B2", ms=5, capsize=2.5, lw=1.2)
        ticks.append(yv); labels.append(f"{'Other or unknown' if lev == 'Other/Unknown' else lev} ({int(r.no_bp_n):,})"); yv += 1
    yv += 0.4
ax.axvline(overall, color="0.5", ls="--", lw=1)
ax.text(overall + 0.01, -0.6, f"Overall, {overall:.2f}", fontsize=8.5, color="0.35", va="center")
ax.set_yticks(ticks); ax.set_yticklabels(labels); ax.set_ylim(yv - 0.4, -1.1); ax.set_xlim(0, 0.7)
ax.set_xlabel("Share of patients with no recorded BP who were flagged"); clean(ax); plt.tight_layout()
save_fig(fig, "figure2b_detection")

# Figure S1: calibration by decile (test set)
calb = rc("figureS1_calibration")
fig, ax = plt.subplots(figsize=(4.2, 4.2))
ax.plot([0, 1], [0, 1], color="0.5", ls="--", lw=1)
for m in MODELS:
    c = calb[calb.model == m]; ax.plot(c.pred, c.obs, "o-", color=COL[m], lw=1.5, ms=4.5, label=LABEL[m])
ax.set_xlim(0, 0.8); ax.set_ylim(0, 0.8)
ax.set_xlabel("Mean predicted risk"); ax.set_ylabel("Observed share with no recorded BP")
clean(ax); ax.legend(frameon=False, fontsize=8.5, loc="upper left"); plt.tight_layout()
save_fig(fig, "figureS1_calibration")

## 6 · Figure 3, Figure S2, Figure S3: association

In [ ]:
CC = {"Minimal Contact": "#999999", "Declining Contact": "#56B4E9", "Rising Contact": "#E69F00",
      "Sustained Contact": "#0072B2", "Sustained Moderate Contact": "#009E73", "Sustained High Contact": "#0072B2"}
f3a, f3b, f3cd = rc("figure3a_trajectories"), rc("figure3b"), rc("figure3cd")
gray = J6["gray"]["class"]

fig, axs = plt.subplots(2, 2, figsize=(7.2, 6.4))
ax = axs[0, 0]
for g in ORDER:
    d = f3a[f3a.label == g].sort_values("q")
    ax.fill_between(d.q, d.lo, d.hi, color=CC[g], alpha=0.2, lw=0); ax.plot(d.q, d.days, "o-", color=CC[g], ms=3.5, lw=1.5, label=g)
ax.set_yscale("log"); ax.set_xticks(range(1, 8)); ax.set_xlabel("Quarter after index"); ax.set_ylabel("Unique contact days per quarter")
ax.legend(frameon=False, fontsize=7)

ax = axs[0, 1]; left = np.zeros(2)
for g in ORDER:
    v = f3b[f3b.label == g].set_index("gap365").pct.reindex([0, 1]).values
    ax.barh([1, 0], v, left=left, color=CC[g], height=0.6); left += v
ax.set_yticks([1, 0]); ax.set_yticklabels(["No gap >365 days", "Gap >365 days"]); ax.set_xlim(0, 100)
ax.set_xlabel("Patients, %")

for ax, cause, lab, p in [(axs[1, 0], 1, "Cardiovascular hospitalization, %", gray[0]),
                          (axs[1, 1], 2, "Death without CV hospitalization, %", gray[1])]:
    for g in ORDER:
        d = f3cd[(f3cd.cause == cause) & (f3cd.label == g)]
        ax.step(d.day, d.cif_pct, where="post", color=CC[g], lw=1.5)
    ax.set_xlim(0, 730); ax.set_xticks([0, 182, 365, 547, 730])
    ax.set_xlabel("Days after the landmark"); ax.set_ylabel(lab)
    ax.text(0.03, 0.95, f"Gray's test p = {p:.2f}", transform=ax.transAxes, fontsize=8, va="top")
for ax, l in zip(axs.flat, "ABCD"):
    clean(ax); ax.text(-0.18, 1.04, l, transform=ax.transAxes, fontsize=12, fontweight="bold")
plt.tight_layout(); save_fig(fig, "figure3")

# Figure S2: gap spline with histogram
cur = rc("figureS2_gap_spline"); kn = json.load(open(os.path.join(RESULTS_DIR, "spline_knots.json")))["gap_days"]
fig, ax = plt.subplots(figsize=(5.5, 3.8)); ax2 = ax.twinx()
ax2.hist(A.gap_days, bins=np.arange(0, 735, 15), color="0.85"); ax2.set_yticks([]); ax2.spines[["top", "right"]].set_visible(False)
ax.set_zorder(ax2.get_zorder() + 1); ax.patch.set_visible(False)
ax.fill_between(cur.gap_days, cur.lo, cur.hi, color="#0072B2", alpha=0.2, lw=0); ax.plot(cur.gap_days, cur.HR, color="#0072B2", lw=1.8)
ax.axhline(1, color="0.3", lw=0.8)
for x in (180, 270, 365): ax.axvline(x, color="0.5", ls=":", lw=1)
ax.set_xlim(0, 730); ax.set_xlabel("Longest interval without recorded contact, days"); ax.set_ylabel("Hazard ratio (vs 90 days)")
ax.text(0.02, 0.95, f"overall p = {J6['gap_spline']['p_overall']:.2f}; knots {', '.join(f'{k:.0f}' for k in kn)}",
        transform=ax.transAxes, fontsize=8, va="top")
clean(ax); plt.tight_layout(); save_fig(fig, "figureS2_gap_spline")

# Figure S3: forest plot by outcome
fs3 = rc("figureS3"); outs = list(dict.fromkeys(fs3.outcome)); terms = list(dict.fromkeys(fs3.term))
fig, axs = plt.subplots(1, len(outs), figsize=(8, 3.2), sharey=True)
for ax, o in zip(np.atleast_1d(axs), outs):
    d = fs3[fs3.outcome == o].set_index("term").reindex(terms)
    yy = np.arange(len(terms))[::-1]
    ax.errorbar(d.HR, yy, xerr=[d.HR - d.lo, d.hi - d.HR], fmt="o", color="#0072B2", ms=4, capsize=2, lw=1)
    ax.axvline(1, color="0.4", lw=0.8); ax.set_xscale("log"); ax.set_title(o, fontsize=9); clean(ax)
np.atleast_1d(axs)[0].set_yticks(np.arange(len(terms))[::-1]); np.atleast_1d(axs)[0].set_yticklabels(terms)
fig.supxlabel("Hazard ratio (95% CI)", fontsize=9); plt.tight_layout(); save_fig(fig, "figureS3_components")

## 7 · Save tables and key numbers

In [ ]:
table1.to_csv(os.path.join(RESULTS_DIR, "table1.csv"), index=False)
tableS4.to_csv(os.path.join(RESULTS_DIR, "tableS4.csv"), index=False)
table3.to_csv(os.path.join(RESULTS_DIR, "table3_with_ci.csv"), index=False)
tableS9.to_csv(os.path.join(RESULTS_DIR, "tableS9_denominators.csv"), index=False)

# every paper table in one workbook, for writing the draft
sheets = {"Table 1": table1, "Table 2A": rc("table2a_discrimination"), "Table 2B": rc("table2b_capacity"),
          "Table 3": table3, "Table 4": rc("table4"), "Table S2A": rc("tableS2a_fit"), "Table S2B": rc("tableS2b_classes"),
          f"Table S2B {J5['alternative']['k']} classes": rc(f"tableS2b_classes_k{J5['alternative']['k']}"),
          "Table S3": rc("tableS3"), "Table S4": tableS4, "Table S5A": rc("tableS5a_alternative_targets"),
          "Table S5B": rc("tableS5b_competing_death"), "Table S6": rc("tableS6_group_thresholds"), "Table S7": rc("tableS7"),
          "Table S8A": rc("tableS8a_components"), "Table S8B": rc("tableS8b_heart_failure"), "Table S9": tableS9,
          "Multimorbidity": rc("multimorbidity_phenotypes")}
with pd.ExcelWriter(os.path.join(RESULTS_DIR, "paper_tables.xlsx")) as xw:
    for k, v in sheets.items():
        v.to_excel(xw, sheet_name=k, index=False)

save_json({
    "prediction_sample": {"n": int(len(S)), "no_bp": int(S.y.sum()), "median_days_index_to_prediction": PRED_DAY},
    "qualifying_reading": {"sbp_mean": round(float(S.q_sbp.mean()), 1), "dbp_mean": round(float(S.q_dbp.mean()), 1),
                           "min_sbp": int(S.q_sbp.min()), "pct_dbp_ge90": round(float(100 * (S.q_dbp >= 90).mean()), 1)},
    "no_earlier_readings": {"n": int(S.e_sbp.isna().sum()), "pct": round(float(100 * S.e_sbp.isna().mean()), 1)},
    "earlier_dbp_only_elevated_after_index": {"n": int(np.isin(ids, early_dbp).sum()),
                                              "pct": round(float(100 * np.isin(ids, early_dbp).mean()), 1)},
    "competing_90": {"bp": int(bp90.sum()), "death_before_bp": int(d90.sum()), "coverage_ended_first": int(cens.sum()),
                     "none": int((~bp90 & ~d90 & ~cens).sum())},
    "figure1_landmark_exclusions": LM_EXCL,
    "day730_sample_also_in_primary": overlap,
}, RESULTS_DIR, "08_key_numbers")
print("figures:", sorted(os.listdir(FIG_DIR)))